# Preparação dos dados de área ardida

* Organizar, filtrar e transformar esta informação para que possa ser usada no cálculo da susceptibilidade, da probabilidade e na validação dos resultados.

In [ ]:
import os
import shutil
from glob import glob

import geopandas as gpd
import numpy as np
import rasterio as rio

from glass.gp.ovl.clipp import clip
from glass.rst.stats import count_region_in_shape

In [ ]:
raw_shps = sorted(glob("/code/data/raw/area_ardida/icnf/*/*.shp")) 

aoi = "/code/data/processed/pnse/aoi/pnse.shp"
ref = "/code/data/processed/pnse/topo/derived/dem_pnse.tif" 

out_year = "/code/data/processed/pnse/area_ardida/yearly_vector" 
out_aoi = "/code/data/processed/pnse/area_ardida/yearly_aoi"
out_train = "/code/data/processed/pnse/area_ardida/train"
out_valid = "/code/data/processed/pnse/area_ardida/valid"

out_rst_count_dir = "/code/data/processed/pnse/area_ardida/raster_count"
out_rst_bin_dir = "/code/data/processed/pnse/area_ardida/raster_binary"
out_tmp_periods = "/code/data/scratch/tmp_periods/pnse/area_ardida"

out_rst = os.path.join(out_rst_count_dir, "rst_ba_1975_2023.tif")
validation_year = 2024

for p in [
    out_year, out_aoi, out_train, out_valid,
    out_rst_count_dir, out_rst_bin_dir, out_tmp_periods
]:
    os.makedirs(p, exist_ok=True)

print("Blocos encontrados:")
for f in raw_shps:
    print("-", f)

In [ ]:
gdf_test = gpd.read_file(raw_shps[0])

print(gdf_test.shape)
print(gdf_test.crs)
print(gdf_test.columns.tolist())
print(sorted(gdf_test["Ano"].dropna().unique())[:10])
print(gdf_test["Ano"].value_counts(dropna=False).sort_index().head())

## Filtragem por área mínima

Aplica-se um filtro por área mínima aos perímetros de incêndio, de modo a manter maior consistência temporal no inventário de áreas ardidas.

In [ ]:
# Aplica-se um filtro por área mínima aos perímetros de incêndio
def filtrar_area_minima(gdf, campo_ano="Ano", campo_area_ha="AreaHaSIG"):
    gdf = gdf.copy()
    gdf = gdf[
        ((gdf[campo_ano] <= 1983) & (gdf[campo_area_ha] > 30)) |
        ((gdf[campo_ano] > 1983) & (gdf[campo_area_ha] > 5))
    ]
    return gdf

## Separação dos blocos de área ardida por ano
* dados do ICNF têm as áreas ardidas agregadas por blocos desde 1975 até 2008, sendo necessária a sua separação

In [ ]:
#exportar os ficheiros anuais separados
for shp in raw_shps:
    gdf = gpd.read_file(shp)
    n0 = len(gdf)

    gdf = filtrar_area_minima(gdf, campo_ano="Ano", campo_area_ha="AreaHaSIG")
    n1 = len(gdf)

    keep_cols = ["Ano", "AreaHaSIG", "geometry"]
    keep_cols = [c for c in keep_cols if c in gdf.columns]
    gdf = gdf[keep_cols].copy()

    anos = sorted(gdf["Ano"].dropna().unique())

    print(f"\nBloco: {os.path.basename(shp)}")
    print(f"Feições antes do filtro: {n0}")
    print(f"Feições depois do filtro: {n1}")
    print("Anos após filtro:", anos)

    for ano in anos:
        ay = gdf[gdf["Ano"] == ano].copy()
        of = os.path.join(out_year, f"aa_{int(ano)}.shp")
        ay.to_file(of)
        print("gravado:", of, "| n =", len(ay))

## Recorte das áreas ardidas anuais à AOI

In [ ]:
year_shps = sorted(glob(os.path.join(out_year, "*.shp")))

print("Ficheiros anuais:", len(year_shps))

for shp in year_shps:
    of = os.path.join(out_aoi, os.path.basename(shp))

    clip(
        inFeat=shp,
        clipFeat=aoi,
        outFeat=of,
        api_gis="ogr2ogr"
    )

    gdf_clip = gpd.read_file(of)
    print("clip:", os.path.basename(of), "| n =", len(gdf_clip))

## Separação Treino/Validação

In [ ]:
aoi_shps = sorted(glob(os.path.join(out_aoi, "*.shp")))

for shp in aoi_shps:
    ano = int(os.path.basename(shp).replace("aa_", "").replace(".shp", ""))

    gdf = gpd.read_file(shp)

    if ano == validation_year:
        dst = os.path.join(out_valid, os.path.basename(shp))
        gdf.to_file(dst)
    else:
        dst = os.path.join(out_train, os.path.basename(shp))
        gdf.to_file(dst)

print("Ficheiros de treino:", len(glob(os.path.join(out_train, "*.shp"))))
print("Ficheiros de validação:", len(glob(os.path.join(out_valid, "*.shp"))))

## Criação dos rasters acumulados

In [ ]:
# Converte células sem ocorrência para nodata no raster final
def zeros_para_nodata(path, nodata=-1):
    with rio.open(path) as src:
        arr = src.read(1)
        profile = src.profile.copy()

    arr = arr.astype("int16")
    arr[arr == 0] = nodata

    profile.update(dtype="int16", nodata=nodata)

    with rio.open(path, "w", **profile) as dst:
        dst.write(arr, 1)

In [ ]:
# Agrega os ficheiros anuais de um dado período e gera um raster de contagem por célula
def criar_raster_ba_periodo(
    anos,
    pasta_origem,
    pasta_tmp_base,
    ref,
    out_raster
):
    nome_periodo = f"{min(anos)}_{max(anos)}"
    pasta_tmp = os.path.join(pasta_tmp_base, nome_periodo)

    if os.path.exists(pasta_tmp):
        shutil.rmtree(pasta_tmp)
    os.makedirs(pasta_tmp, exist_ok=True)

    for ano in anos:
        shp = os.path.join(pasta_origem, f"aa_{ano}.shp")
        if os.path.exists(shp):
            for f in glob(shp.replace(".shp", ".*")):
                shutil.copy(f, pasta_tmp)

    count_region_in_shape(
        folder=pasta_tmp,
        ref=ref,
        out=out_raster,
        returnprob=None
    )

    zeros_para_nodata(out_raster)

    print("Raster criado:", out_raster)

In [ ]:
# raster acumulado geral
criar_raster_ba_periodo(
    anos=list(range(1975, 2024)),
    pasta_origem=out_train,
    pasta_tmp_base=out_tmp_periods,
    ref=ref,
    out_raster=os.path.join(out_rst_count_dir, "rst_ba_1975_2023.tif")
)

# raster acumulado compatível com LULC
criar_raster_ba_periodo(
    anos=list(range(1995, 2024)),
    pasta_origem=out_train,
    pasta_tmp_base=out_tmp_periods,
    ref=ref,
    out_raster=os.path.join(out_rst_count_dir, "rst_ba_1995_2023.tif")
)

# janelas temporais 
criar_raster_ba_periodo(
    anos=list(range(1995, 2007)),
    pasta_origem=out_aoi,
    pasta_tmp_base=out_tmp_periods,
    ref=ref,
    out_raster=os.path.join(out_rst_count_dir, "rst_ba_1995_2006.tif")
)

criar_raster_ba_periodo(
    anos=list(range(2007, 2010)),
    pasta_origem=out_aoi,
    pasta_tmp_base=out_tmp_periods,
    ref=ref,
    out_raster=os.path.join(out_rst_count_dir, "rst_ba_2007_2009.tif")
)

criar_raster_ba_periodo(
    anos=list(range(2010, 2015)),
    pasta_origem=out_aoi,
    pasta_tmp_base=out_tmp_periods,
    ref=ref,
    out_raster=os.path.join(out_rst_count_dir, "rst_ba_2010_2014.tif")
)

criar_raster_ba_periodo(
    anos=list(range(2015, 2018)),
    pasta_origem=out_aoi,
    pasta_tmp_base=out_tmp_periods,
    ref=ref,
    out_raster=os.path.join(out_rst_count_dir, "rst_ba_2015_2017.tif")
)


criar_raster_ba_periodo(
    anos=list(range(2018, 2024)),
    pasta_origem=out_aoi,
    pasta_tmp_base=out_tmp_periods,
    ref=ref,
    out_raster=os.path.join(out_rst_count_dir, "rst_ba_2018_2023.tif")
)

In [ ]:
print("Anuais separados:", len(glob(os.path.join(out_year, "*.shp"))))
print("Anuais recortados à AOI:", len(glob(os.path.join(out_aoi, "*.shp"))))
print("Anuais de treino:", len(glob(os.path.join(out_train, "*.shp"))))
print("Anuais de validação:", len(glob(os.path.join(out_valid, "*.shp"))))

## Criação dos rasters binários

In [ ]:
def contagem_para_binario(src_path, dst_path, nodata=-1):
    with rio.open(src_path) as src:
        arr = src.read(1)
        profile = src.profile.copy()
        src_nodata = src.nodata

    out = np.full(arr.shape, nodata, dtype="int16")

    if src_nodata is None:
        mask = arr > 0
    else:
        mask = arr != src_nodata

    out[mask] = 1

    profile.update(dtype="int16", nodata=nodata)

    with rio.open(dst_path, "w", **profile) as dst:
        dst.write(out, 1)

    print("Raster binário criado:", dst_path)

In [ ]:
criar_raster_ba_periodo(
    anos=[2024],
    pasta_origem=out_valid,
    pasta_tmp_base=out_tmp_periods,
    ref=ref,
    out_raster=os.path.join(out_rst_count_dir, "rst_ba_2024.tif")
)

In [ ]:
contagem_para_binario(
    os.path.join(out_rst_count_dir, "rst_ba_1995_2023.tif"),
    os.path.join(out_rst_bin_dir, "rst_ba_1995_2023_bin.tif")
)

contagem_para_binario(
    os.path.join(out_rst_count_dir, "rst_ba_2024.tif"),
    os.path.join(out_rst_bin_dir, "rst_ba_2024_bin.tif")
)